# Scam Detector Evaluation against Kaggle Fake Job Postings Dataset

This notebook evaluates Job Raider's rule-based scam detector against the Kaggle
"Fake Job Posting Prediction" dataset (~18K labeled postings). We measure precision,
recall, F1, find the optimal confidence threshold, and analyze which indicators
correlate most strongly with ground-truth fraudulent postings.

**Author:** Job Raider
**Date:** 2026-04-22

## 1. Setup

Add your Kaggle credentials to `backend-py/.env` (see `.env.example`), then run this cell to download the dataset automatically.

In [ ]:
import sys
import os
from pathlib import Path

# Ensure imports resolve from backend-py root
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == "notebooks":
    BACKEND_DIR = NOTEBOOK_DIR.parent
else:
    BACKEND_DIR = NOTEBOOK_DIR

sys.path.insert(0, str(BACKEND_DIR))

# Load credentials from backend-py/.env into the environment
env_path = BACKEND_DIR / ".env"
if env_path.exists():
    from dotenv import load_dotenv
    load_dotenv(env_path)
    print(f"Loaded environment from {env_path}")
else:
    print(f"Warning: No .env file found at {env_path}")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score,
    confusion_matrix, classification_report, precision_recall_curve,
)
from collections import Counter

from src.scoring.scam_detector import JobScamDetector, ScamReport, ScamIndicator
from src.models.job_listing import (
    JobListing, JobSource, JobRequirement, SalaryRange, Skill,
    ExperienceLevel, JobType, WorkMode,
)

print("All imports successful.")
print(f"Backend directory: {BACKEND_DIR}")

## 2. Load Dataset

In [ ]:
# Locate the CSV
csv_path = BACKEND_DIR / "data" / "kaggle" / "fake_job_postings.csv"

if not csv_path.exists():
    print("Dataset not found locally. Downloading from Kaggle...")
    import subprocess
    result = subprocess.run(
        ["kaggle", "datasets", "download", "-d",
         "shivamb/real-or-fake-fake-job-posting-prediction",
         "-p", str(BACKEND_DIR / "data" / "kaggle"), "--unzip"],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"Kaggle download failed:\n{result.stderr}\n\n"
            "Make sure KAGGLE_USERNAME and KAGGLE_KEY are set in backend-py/.env"
        )
    print("Download complete.")

df = pd.read_csv(csv_path)
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nClass distribution:")
print(df["fraudulent"].value_counts().to_string())
print(f"\nFraudulent ratio: {df['fraudulent'].mean():.2%}")
df.head(3)

## 3. Dataset Exploration

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Class balance
ax = axes[0]
df["fraudulent"].value_counts().plot(kind="bar", ax=ax, color=["#2ecc71", "#e74c3c"])
ax.set_xticklabels(["Legitimate (0)", "Fraudulent (1)"], rotation=0)
ax.set_title("Class Distribution")
ax.set_ylabel("Count")

# Missing values
ax = axes[1]
missing = df.isnull().mean().sort_values(ascending=False)
missing[missing > 0].plot(kind="barh", ax=ax, color="#e67e22")
ax.set_title("Missing Value Ratio by Column")
ax.set_xlabel("Fraction Missing")

# Text length by class
ax = axes[2]
df["desc_len"] = df["description"].fillna("").str.len()
df.boxplot(column="desc_len", by="fraudulent", ax=ax)
ax.set_title("Description Length by Class")
ax.set_xlabel("Fraudulent")
ax.set_ylabel("Characters")
fig.suptitle("")

plt.tight_layout()
plt.show()

## 4. Convert Kaggle Rows to JobListing Objects

In [ ]:
def parse_salary_range(salary_str: str) -> SalaryRange | None:
    """Parse Kaggle salary_range string (e.g. '50000-80000') into SalaryRange."""
    if pd.isna(salary_str) or not salary_str:
        return None
    parts = str(salary_str).split("-")
    if len(parts) == 2:
        try:
            return SalaryRange(
                min_amount=float(parts[0].strip()),
                max_amount=float(parts[1].strip()),
                period="annual",
            )
        except ValueError:
            return None
    return None


EXPERIENCE_MAP = {
    "Internship": ExperienceLevel.INTERNSHIP,
    "Entry level": ExperienceLevel.ENTRY,
    "Mid-Senior level": ExperienceLevel.MID,
    "Senior": ExperienceLevel.SENIOR,
    "Director": ExperienceLevel.LEAD,
    "Executive": ExperienceLevel.EXECUTIVE,
    "Not Applicable": ExperienceLevel.NOT_SPECIFIED,
}

JOB_TYPE_MAP = {
    "Full-time": JobType.FULL_TIME,
    "Part-time": JobType.PART_TIME,
    "Contract": JobType.CONTRACT,
    "Temporary": JobType.TEMPORARY,
    "Internship": JobType.INTERNSHIP,
    "Other": JobType.FREELANCE,
}


def row_to_job_listing(row: pd.Series) -> JobListing:
    """Convert a Kaggle dataset row to a JobListing object."""
    # Combine description and company_profile for richer text analysis
    desc_parts = []
    if pd.notna(row.get("description")):
        desc_parts.append(str(row["description"]))
    if pd.notna(row.get("company_profile")):
        desc_parts.append(str(row["company_profile"]))
    combined_desc = "\n\n".join(desc_parts) if desc_parts else None

    # Parse requirements into JobRequirement objects
    requirements = []
    if pd.notna(row.get("requirements")):
        req_text = str(row["requirements"])
        for line in req_text.split("\n"):
            line = line.strip()
            if line:
                requirements.append(JobRequirement(text=line))

    return JobListing(
        title=str(row.get("title", "Unknown")),
        company=str(row.get("company", "Unknown")) if pd.notna(row.get("company")) else "Unknown",
        location=str(row.get("location")) if pd.notna(row.get("location")) else None,
        description=combined_desc,
        requirements=requirements,
        salary_range=parse_salary_range(row.get("salary_range")),
        source=JobSource.MANUAL,
        job_id=str(row.get("job_id", "")),
        is_remote=bool(row.get("telecommuting", 0)),
        experience_level=EXPERIENCE_MAP.get(
            str(row.get("required_experience", "")), ExperienceLevel.NOT_SPECIFIED
        ),
        job_type=JOB_TYPE_MAP.get(str(row.get("employment_type", "")), JobType.FULL_TIME),
        metadata={
            "has_company_logo": bool(row.get("has_company_logo", 0)),
            "has_questions": bool(row.get("has_questions", 0)),
            "industry": str(row.get("industry", "")) if pd.notna(row.get("industry")) else None,
            "function": str(row.get("function", "")) if pd.notna(row.get("function")) else None,
            "benefits": str(row.get("benefits", "")) if pd.notna(row.get("benefits")) else None,
        },
    )


# Convert all rows
print("Converting Kaggle rows to JobListing objects...")
job_listings = [row_to_job_listing(row) for _, row in df.iterrows()]
ground_truth = df["fraudulent"].values

print(f"Converted {len(job_listings)} listings.")
print(f"Fraudulent: {sum(ground_truth)}, Legitimate: {len(ground_truth) - sum(ground_truth)}")

## 5. Run Scam Detector

In [ ]:
detector = JobScamDetector(threshold=0.7)
print(f"Running scam detection on {len(job_listings)} listings (threshold={detector.threshold})...")

reports = detector.batch_detect(job_listings)

# Extract predictions and confidence scores
predictions = np.array([1 if r.is_scam else 0 for r in reports])
confidences = np.array([r.confidence for r in reports])
risk_scores = np.array([r.risk_score for r in reports])

print(f"\nDetector flagged {predictions.sum()} as scam (of {len(ground_truth)} total)")
print(f"Ground truth fraudulent: {ground_truth.sum()}")
print(f"Detector flagged ratio: {predictions.mean():.2%}")
print(f"Actual fraud ratio: {ground_truth.mean():.2%}")

## 6. Metrics Calculation

In [ ]:
precision = precision_score(ground_truth, predictions)
recall = recall_score(ground_truth, predictions)
f1 = f1_score(ground_truth, predictions)
accuracy = accuracy_score(ground_truth, predictions)

print("=" * 50)
print("Scam Detector Performance (threshold=0.7)")
print("=" * 50)
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}  (flagged scams that are actually scams)")
print(f"  Recall:    {recall:.4f}  (actual scams that were caught)")
print(f"  F1 Score:  {f1:.4f}")
print()

print(classification_report(ground_truth, predictions, target_names=["Legitimate", "Fraudulent"]))

# Confusion matrix
cm = confusion_matrix(ground_truth, predictions)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues", ax=ax,
    xticklabels=["Legitimate", "Fraudulent"],
    yticklabels=["Legitimate", "Fraudulent"],
)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix (threshold=0.7)")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"\nTrue Negatives:  {tn:>6}  (correctly identified as legitimate)")
print(f"False Positives: {fp:>6}  (legitimate incorrectly flagged as scam)")
print(f"False Negatives: {fn:>6}  (scams that were missed)")
print(f"True Positives:  {tp:>6}  (correctly identified as scam)")

## 7. Threshold Analysis

Sweep the confidence threshold from 0.0 to 1.0 and find the optimal operating point.

In [ ]:
thresholds = np.arange(0.0, 1.01, 0.05)
precisions = []
recalls = []
f1s = []

for t in thresholds:
    preds_t = (confidences >= t).astype(int)
    if preds_t.sum() == 0:
        precisions.append(0.0)
    else:
        precisions.append(precision_score(ground_truth, preds_t, zero_division=0))
    recalls.append(recall_score(ground_truth, preds_t, zero_division=0))
    f1s.append(f1_score(ground_truth, preds_t, zero_division=0))

best_idx = np.argmax(f1s)
best_threshold = thresholds[best_idx]
best_f1 = f1s[best_idx]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(thresholds, precisions, label="Precision", marker="o", markersize=3)
ax.plot(thresholds, recalls, label="Recall", marker="s", markersize=3)
ax.plot(thresholds, f1s, label="F1 Score", marker="^", markersize=3)
ax.axvline(x=best_threshold, color="red", linestyle="--", alpha=0.7, label=f"Best F1 @ {best_threshold:.2f}")
ax.axvline(x=0.7, color="gray", linestyle=":", alpha=0.7, label="Current threshold (0.70)")
ax.set_xlabel("Confidence Threshold")
ax.set_ylabel("Score")
ax.set_title("Precision / Recall / F1 vs Confidence Threshold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Optimal threshold (max F1): {best_threshold:.2f}")
print(f"  F1:       {f1s[best_idx]:.4f}")
print(f"  Precision: {precisions[best_idx]:.4f}")
print(f"  Recall:    {recalls[best_idx]:.4f}")
print(f"\nCurrent threshold (0.70):")
print(f"  F1:       {f1:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")

## 8. Error Analysis

Examine false negatives (scams we missed) and false positives (legit jobs we flagged).

In [ ]:
# Identify error indices
fn_indices = np.where((ground_truth == 1) & (predictions == 0))[0]  # missed scams
fp_indices = np.where((ground_truth == 0) & (predictions == 1))[0]  # false alarms

print(f"False Negatives (missed scams): {len(fn_indices)}")
print(f"False Positives (false alarms): {len(fp_indices)}")

# Show sample false negatives
print("\n" + "=" * 60)
print("SAMPLE FALSE NEGATIVES (scams the detector missed)")
print("=" * 60)
for i in fn_indices[:5]:
    row = df.iloc[i]
    report = reports[i]
    print(f"\n--- Job ID: {row.get('job_id', 'N/A')} ---")
    print(f"  Title: {row.get('title', 'N/A')}")
    print(f"  Company: {row.get('company', 'N/A')}")
    print(f"  Confidence: {report.confidence:.2f} | Risk Score: {report.risk_score}")
    print(f"  Indicators: {[ind.value for ind in report.indicators]}")
    desc = str(row.get("description", ""))[:200]
    print(f"  Description excerpt: {desc}...")

# Show sample false positives
print("\n" + "=" * 60)
print("SAMPLE FALSE POSITIVES (legit jobs incorrectly flagged)")
print("=" * 60)
for i in fp_indices[:5]:
    row = df.iloc[i]
    report = reports[i]
    print(f"\n--- Job ID: {row.get('job_id', 'N/A')} ---")
    print(f"  Title: {row.get('title', 'N/A')}")
    print(f"  Company: {row.get('company', 'N/A')}")
    print(f"  Confidence: {report.confidence:.2f} | Risk Score: {report.risk_score}")
    print(f"  Reasons: {report.reasons[:3]}")
    desc = str(row.get("description", ""))[:200]
    print(f"  Description excerpt: {desc}...")

## 9. Indicator Breakdown

Which ScamIndicators fire most often, and how do they correlate with ground-truth labels?

In [ ]:
# Count indicator occurrences overall and by class
indicator_counts = {ind: {"legit": 0, "fraud": 0} for ind in ScamIndicator}

for i, report in enumerate(reports):
    label = "fraud" if ground_truth[i] == 1 else "legit"
    for ind in report.indicators:
        indicator_counts[ind][label] += 1

# Build a DataFrame for visualization
ind_data = []
for ind, counts in indicator_counts.items():
    ind_data.append({
        "indicator": ind.value,
        "legit": counts["legit"],
        "fraud": counts["fraud"],
        "total": counts["legit"] + counts["fraud"],
        "fraud_pct": counts["fraud"] / max(counts["fraud"] + counts["legit"], 1) * 100,
    })

ind_df = pd.DataFrame(ind_data).sort_values("total", ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Stacked bar chart
ax = axes[0]
ind_df[["indicator", "legit", "fraud"]].set_index("indicator").plot(
    kind="barh", stacked=True, ax=ax, color=["#2ecc71", "#e74c3c"]
)
ax.set_title("Indicator Frequency (Legit vs Fraud)")
ax.set_xlabel("Count")

# Fraud percentage per indicator
ax = axes[1]
ind_df_sorted = ind_df.sort_values("fraud_pct", ascending=True)
ax.barh(ind_df_sorted["indicator"], ind_df_sorted["fraud_pct"], color="#e74c3c")
ax.set_title("Fraud Percentage When Indicator Fires")
ax.set_xlabel("% that are actually fraudulent")
ax.axvline(x=ground_truth.mean() * 100, color="gray", linestyle=":", label=f"Baseline ({ground_truth.mean()*100:.1f}%)")
ax.legend()

plt.tight_layout()
plt.show()

print("\nIndicator Summary Table:")
print(ind_df.to_string(index=False))

## 10. Summary and Recommendations

In [ ]:
print("=" * 60)
print("SCAM DETECTOR EVALUATION SUMMARY")
print("=" * 60)
print()
print(f"Dataset: {len(df)} job postings ({ground_truth.sum()} fraudulent, {len(ground_truth)-ground_truth.sum()} legitimate)")
print(f"Fraud rate: {ground_truth.mean():.2%}")
print()
print("Performance at default threshold (0.70):")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")
print(f"  Accuracy:  {accuracy:.4f}")
print()
print(f"Optimal threshold: {best_threshold:.2f}")
print(f"  F1 at optimal: {f1s[best_idx]:.4f}")
print(f"  Precision: {precisions[best_idx]:.4f}")
print(f"  Recall:    {recalls[best_idx]:.4f}")
print()

# Top indicators by fraud correlation
top_indicators = ind_df.sort_values("fraud_pct", ascending=False).head(5)
print("Top 5 indicators by fraud correlation:")
for _, row in top_indicators.iterrows():
    print(f"  {row['indicator']:25s} -> {row['fraud_pct']:.1f}% fraud rate ({row['total']} occurrences)")
print()

# Recommendations
print("RECOMMENDATIONS:")
if recall < 0.5:
    print("  - Low recall: consider lowering threshold or adding new indicators")
    print("    to catch more scams. Review false negative samples above.")
if precision < 0.5:
    print("  - Low precision: consider raising threshold or refining indicator")
    print("    patterns to reduce false alarms on legitimate postings.")
if f1 > 0.6:
    print("  - Detector is performing reasonably well for a rule-based system.")
    print("    Consider using risk_score as a feature in an ML ensemble model.")